In [1]:
import pandas as pd
import dask.array as da
from dask_ml.model_selection import GridSearchCV
from dask.distributed import Client, LocalCluster
from dask_ml.model_selection import KFold, train_test_split
# from xgboost.dask import DaskXGBRegressor  # Use xgboost's Dask integration
from dask_ml.xgboost import XGBRegressor
import logging

ModuleNotFoundError: No module named 'dask_xgboost'

### Without grid_search

In [2]:
# Function to perform the 5-fold cross-validation and out-of-sample test
def train_model(client: Client, X: da.Array, y: da.Array, n_splits=5, test_size=0.2):
    
    # Split the data into training and testing sets (Dask-aware)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # Set up DaskXGBRegressor
    regressor  = DaskXGBRegressor(verbosity=1, n_estimators=100, tree_method="hist")

    # Perform 5-fold cross-validation
    kf         = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_results = []

    for train_index, val_index in kf.split(X_train, y_train):
        X_train_cv, X_val_cv = X_train[train_index], X_train[val_index]
        y_train_cv, y_val_cv = y_train[train_index], y_train[val_index]

        print('type(X_train)',type(X_train))
        print('type(X_train_cv)',type(X_train_cv))
        
        # Train the model
        regressor.client = client  # Attach the Dask client
        regressor.fit(X_train_cv, y_train_cv, eval_set=[(X_val_cv, y_val_cv)])
        
        print('finish fit regressor')
        
        # Make predictions and evaluate the model
        predictions = regressor.predict(X_val_cv)
        cv_results.append((predictions, y_val_cv))

        print('finish predictions')
        print('cv_results', cv_results)
        
    # After cross-validation, train on the full training set
    regressor.fit(X_train, y_train, eval_set=[(X_test, y_test)])

    # Evaluate the final model on the test set
    final_predictions = regressor.predict(X_test)
    print("Out-of-sample predictions:", final_predictions.compute())

    return regressor, cv_results


### With grid_search

In [2]:
def train_model(client: Client, X: da.Array, y: da.Array, n_splits=5, test_size=0.2):
    
    # Split the data into training and testing sets (Dask-aware)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # Set up DaskXGBRegressor
    # regressor = DaskXGBRegressor(verbosity=1, tree_method="hist")
    regressor = XGBRegressor(objective="reg:squarederror")
    
    # Define the grid search parameters
    param_grid = {
        'learning_rate': [0.1],#, 0.25, 0.35, 0.5],
        'max_depth': [7],#, 8, 9, 10, 11, 12],
        'subsample': [0.2],#, 0.4, 0.6, 0.8, 1],
        'n_estimators': [100, 150],#, 200, 250, 300, 350, 400]
    }

    # Perform grid search with cross-validation
    grid_search = GridSearchCV(regressor, 
                               param_grid, 
                               cv=KFold(n_splits=n_splits, shuffle=True, random_state=42),
                               scoring="neg_mean_squared_error")
    
    # Train the model using grid search
    grid_search.fit(X_train, y_train)

    # Get the best parameters from the grid search
    best_params = grid_search.best_params_
    print("Best hyperparameters found:", best_params)
    
    # After grid search, use the best model to train on the full training set
    best_regressor = grid_search.best_estimator_
    best_regressor.client = client  # Attach the Dask client
    
    # Evaluate the final model on the test set
    final_predictions = best_regressor.predict(X_test)
    print("Out-of-sample predictions:", final_predictions.compute())

    return best_regressor, grid_search.cv_results_

In [3]:

logging.basicConfig(level=logging.DEBUG)

# Start Dask client and cluster
with LocalCluster(n_workers=4, threads_per_worker=1, memory_limit="4GB") as cluster:
    with Client(cluster) as client:
        
        # Load your dataset
        var_input = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/Qle_all_sites.csv',
                                usecols=['time', 'site_name', 'model_CABLE', 'VPD', 'obs_Tair', 'obs_Precip', 'obs_SWdown'],
                                na_values=[''])
        var_input['CABLE_SMtop1m'] = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/SMtop1m_all_sites.csv',
                                                 usecols=['CABLE_SMtop1m'], na_values=[''])
        var_input['obs_LAI'] = pd.read_csv('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/LAI_all_sites_parallel.csv',
                                           usecols=['obs_LAI'], na_values=[''])

        # Select a site for a trial
        site_name = 'AU-Tum'
        var_select = var_input[var_input['site_name'] == site_name]

        # Extract features and target
        X = var_select[['VPD', 'obs_Tair', 'obs_Precip', 'obs_SWdown', 'CABLE_SMtop1m', 'obs_LAI']].values
        y = var_select['model_CABLE'].values

        # print(var_input)
        # print(var_select)
        
        # Convert to Dask arrays
        X_dask = da.from_array(X, chunks=("auto", X.shape[1]))  # Adjust chunk size as needed
        y_dask = da.from_array(y, chunks="auto")
        
        # print(client)
        # print(client.scheduler_info())  # Check if the scheduler is active
        
        # Train the model and get results
        model, results = train_model(client, X_dask, y_dask)

        # Print cross-validation results
        print("Cross-validation results:", results)


DEBUG:asyncio:Using selector: EpollSelector
2025-03-05 21:25:53,138 - distributed.worker - WARNING - Mismatched versions found

+---------+---------------------------------------------+----------------+----------------+
| Package | Worker-04c81778-a6d8-4656-a762-baa951df8142 | Scheduler      | Workers        |
+---------+---------------------------------------------+----------------+----------------+
| python  | 3.12.2.final.0                              | 3.12.9.final.0 | 3.12.2.final.0 |
+---------+---------------------------------------------+----------------+----------------+
2025-03-05 21:25:53,138 - distributed.worker - WARNING - Mismatched versions found

+---------+---------------------------------------------+----------------+----------------+
| Package | Worker-7f045c01-03a0-45db-9284-19a01637ec7f | Scheduler      | Workers        |
+---------+---------------------------------------------+----------------+----------------+
| python  | 3.12.2.final.0                          

KeyError: 'cv-split-5d67ebb194aff3f848b3e3a06fd7c7ad'